In [2]:
import numpy as np
import pandas as pd

raw = pd.read_csv("../data/PakWheels Dataset.csv")
print(raw.shape)

(61919, 15)


In [3]:
raw.info()
raw.head()

<class 'pandas.DataFrame'>
RangeIndex: 61919 entries, 0 to 61918
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   nam              61919 non-null  str  
 1   Price            61919 non-null  str  
 2   Year             61919 non-null  int64
 3   Millage          61919 non-null  str  
 4   Fuel             61919 non-null  str  
 5   Transmission     61919 non-null  str  
 6   Province         61919 non-null  str  
 7   Color            61918 non-null  str  
 8   Assembly         61919 non-null  str  
 9   Engine Capacity  61919 non-null  str  
 10  Body Type        54732 non-null  str  
 11  Ad Reference     61919 non-null  int64
 12  Features         58042 non-null  str  
 13  Owner nam        53493 non-null  str  
 14  url              61919 non-null  str  
dtypes: int64(2), str(13)
memory usage: 27.5 MB


,nam,Price,Year,Millage,Fuel,Transmission,Province,Color,Assembly,Engine Capacity,Body Type,Ad Reference,Features,Owner nam,url
0,Suzuki Alto VXL AGS 2022,PKR 23.9 lacs,2022,"120,000 km",Petrol,Automatic,Islamabad,Solid White,Local,660 cc,Hatchback,10313920,"['ABS', 'AM/FM Radio', 'Air Bags', 'Air Condit...",Daniyal,https://www.pakwheels.com//used-cars/suzuki-al...
1,Honda N Box Custom GL 2022,PKR 33.4 lacs,2022,"37,110 km",Petrol,Automatic,Islamabad,White,Imported,658 cc,Hatchback,10233745,"['ABS', 'AM/FM Radio', 'Air Bags', 'Air Condit...",NaN,https://www.pakwheels.com//used-cars/honda-n-b...
2,Honda Civic EX 1995,PKR 6.3 lacs,1995,786 km,LPG,Manual,Karachi,Black,Local,1500 cc,Sedan,10313884,"['AM/FM Radio', 'Air Conditioning', 'Cassette ...",Abdullah Sheikh,https://www.pakwheels.com//used-cars/honda-civ...
3,Toyota Corolla GLi Automatic 1.6 VVTi 2012,PKR 33.25 lacs,2012,"133,000 km",Petrol,Automatic,Lahore,Medium Silver,Imported,1600 cc,Sedan,10311918,"['ABS', 'AM/FM Radio', 'Air Conditioning', 'CD...",Zakiulhassan,https://www.pakwheels.com//used-cars/toyota-co...
4,Toyota Corolla Hatchback 1998,PKR 16.45 lacs,1998,"125,225 km",Petrol,Automatic,Punjab,Black,Local,1600 cc,NaN,10207521,"['ABS', 'AM/FM Radio', 'Air Bags', 'Air Condit...",NaN,https://www.pakwheels.com//used-cars/toyota-co...


In [4]:
print(raw.isnull().sum())
print("Duplicate rows:", raw.duplicated().sum())
print("Duplicate Ad References:", raw["Ad Reference"].duplicated().sum())
raw.describe()

nam                   0
Price                 0
Year                  0
Millage               0
Fuel                  0
Transmission          0
Province              0
Color                 1
Assembly              0
Engine Capacity       0
Body Type          7187
Ad Reference          0
Features           3877
Owner nam          8426
url                   0
dtype: int64
Duplicate rows: 0
Duplicate Ad References: 8


,Year,Ad Reference
count,61919.000000,6.191900e+04
mean,2012.798026,1.021495e+07
std,9.000544,2.196495e+05
min,1900.000000,1.955064e+06
25%,2007.000000,1.021720e+07
50%,2015.000000,1.025864e+07
75%,2020.000000,1.029217e+07
max,2025.000000,1.031861e+07


In [5]:
# 1) Price mein kaun si units hain?
print(raw["Price"].str.extract(r"(lacs|crore)\s*$")[0].value_counts(dropna=False))
print(raw.loc[~raw["Price"].str.contains("lacs|crore"), "Price"].value_counts().head())

# 2) Mileage ko number bana kar dekho kitni extreme hai
km = pd.to_numeric(raw["Millage"].str.replace(r"[^\d]", "", regex=True), errors="coerce")
print(km.describe())
print("Mileage < 1000 km:", (km < 1000).sum(), "| > 500,000 km:", (km > 500_000).sum())

# 3) Galat saal
print("Year < 1970:", (raw["Year"] < 1970).sum())

0
lacs     57713
crore     3353
NaN        853
Name: count, dtype: int64
Price
Call for price    853
Name: count, dtype: int64
count      61919.000000
mean       99841.268044
std        92636.581567
min            1.000000
25%        44000.000000
50%        87254.000000
75%       131000.000000
max      1000000.000000
Name: Millage, dtype: float64
Mileage < 1000 km: 2562 | > 500,000 km: 405
Year < 1970: 34


In [7]:
import sys
sys.path.append("../src")
import cleaning

print(cleaning.parse_price("PKR 1.27 crore"))
print(cleaning.parse_price("PKR 31.25 lacs"))
print(cleaning.parse_price("Call for price"))
print(cleaning.to_number(pd.Series(["120,000 km", "786 km", "15 km"])))

12700000.0
3125000.0
nan
0    120000
1       786
2        15
dtype: int64


In [8]:
df_clean = cleaning.clean_data(raw)
print(raw.shape, "->", df_clean.shape)
print(df_clean.isnull().sum()[lambda s: s > 0])
df_clean.head()

(61919, 15) -> (61024, 14)
mileage_km    1649
engine_cc       66
dtype: int64


,price,make,model,year,car_age,mileage_km,engine_cc,fuel,transmission,assembly,body_type,location,is_registered,n_features
0,2390000.0,Suzuki,Alto,2022,3,120000.0,660.0,Petrol,Automatic,Local,Hatchback,Islamabad,1,10
1,3340000.0,Honda,N Box,2022,3,37110.0,658.0,Petrol,Automatic,Imported,Hatchback,Islamabad,1,20
2,630000.0,Honda,Civic,1995,30,NaN,1500.0,LPG,Manual,Local,Sedan,Karachi,1,8
3,3325000.0,Toyota,Corolla,2012,13,133000.0,1600.0,Petrol,Automatic,Imported,Sedan,Lahore,1,9
4,1645000.0,Toyota,Corolla,1998,27,125225.0,1600.0,Petrol,Automatic,Local,Unknown,Punjab,1,15


In [9]:
df_clean.to_csv("../data/cleaned_cars.csv", index=False)